# Fase 4. Notebook final: análisis de morosidad en patentes por no uso de aguas
**MCDI500 · Grupo 9** (Christian Abarca Araya, Vicente Agosin Orellana, Patricia Campos Tapia, Daniel Valdivia Tapia) · Docente: Omar Salinas Silva

Este notebook cierra el flujo F1 a F4. Usa el procesamiento de la Fase 2 (reproducido por las clases de la Fase 3) y responde la pregunta técnica del proyecto: **¿cómo se distribuye la morosidad según el año de la patente, la Tesorería Regional, el tipo de titular y el monto?**
Los resultados describen asociaciones; no establecen causas.

**Declaración de uso de IA.** Este notebook se elaboró con apoyo de IA generativa (Anthropic, 2026). Las cifras salen de la ejecución de este mismo archivo; las decisiones y la revisión son responsabilidad del grupo.

## 1. Entorno y datos

In [ ]:
import sys, platform, subprocess
from pathlib import Path
import numpy as np, pandas as pd, matplotlib, sklearn
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(RAIZ / "src"))
from indicadores import cargar_base_f4, tasa_con_intervalo, intervalo_wilson
print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| matplotlib", matplotlib.__version__)
df = cargar_base_f4()
print(f"Filas: {len(df):,} | columnas: {df.shape[1]} | nulos: {int(df.isna().sum().sum())}".replace(",", "."))
print(f"Morosos: {int(df.moroso.sum()):,} ({df.moroso.mean():.2%})".replace(",", "."))
assert df.shape[0] == 46433 and df.isna().sum().sum() == 0

Se parte de los conjuntos de entrenamiento y prueba de la Fase 3 (37.146 y 9.287 filas), que juntos reproducen el archivo procesado de la Fase 2 (46.433 filas). La tesorería y el tipo de persona se reconstruyen desde las columnas *one-hot*.
`ANIO_PATENTE = 0` marca las 691 filas sin año; **no es un año real**, por lo que se excluyen de los análisis por año y se muestran aparte.

## 2. Morosidad según el año de la patente

In [ ]:
con_anio = df[df.anio_faltante == 0]
por_anio = tasa_con_intervalo(con_anio, "ANIO_PATENTE")
print(por_anio.assign(tasa=lambda d: (d.tasa*100).round(1), inf=lambda d: (d.inf*100).round(1), sup=lambda d: (d.sup*100).round(1)).to_string(index=False))
sin_anio = df[df.anio_faltante == 1]
print(f"\nSin año: n={len(sin_anio)} | tasa={sin_anio.moroso.mean():.1%}")
for nombre, m in [("2020 o antes", con_anio.ANIO_PATENTE <= 2020), ("2021 o después", con_anio.ANIO_PATENTE >= 2021)]:
    s = con_anio[m]; lo, hi = intervalo_wilson(s.moroso.sum(), len(s))
    print(f"{nombre}: n={len(s):,} | tasa={s.moroso.mean():.1%} (IC 95 % {lo:.1%} a {hi:.1%})".replace(",", "."))

**Lectura.** La tasa ronda 90 % para patentes hasta 2020 y baja a cerca de 65 % desde 2021, con un descenso brusco entre 2020 y 2021. Dentro de 2021 a 2025 la tasa se mantiene estable (entre 63 % y 67 %), por lo que **no se observa un aumento progresivo hacia los años recientes** que sugiera que el saldo se explique solo por patentes aún no vencidas. La causa del cambio de 2021 no se puede determinar con estos datos.

## 3. Morosidad por Tesorería Regional y tipo de titular

In [ ]:
por_teso = tasa_con_intervalo(df, "tesoreria").sort_values("tasa", ascending=False)
print(por_teso.assign(tasa=lambda d: (d.tasa*100).round(1), inf=lambda d: (d.inf*100).round(1), sup=lambda d: (d.sup*100).round(1)).to_string(index=False))
por_persona = tasa_con_intervalo(df, "persona")
print("\n", por_persona.assign(tasa=lambda d: (d.tasa*100).round(1), inf=lambda d: (d.inf*100).round(1), sup=lambda d: (d.sup*100).round(1)).to_string(index=False))
print("\nTitulares jurídicos con patente 2021 o posterior:", f"{(con_anio[con_anio.persona=='jurídica'].ANIO_PATENTE>=2021).mean():.1%}",
      "| naturales:", f"{(con_anio[con_anio.persona=='natural'].ANIO_PATENTE>=2021).mean():.1%}")

**Lectura.** La tasa va desde 4,7 % (Tarapacá, n = 107) hasta 90,7 % (Araucanía, n = 10.682). Los intervalos de las tesorerías con pocos cobros (Tarapacá, Arica y Parinacota) son anchos y deben leerse con cautela. Los titulares naturales tienen una tasa más alta (84,3 %) que los jurídicos (72,6 %); esta diferencia puede estar influida por el año de la patente, porque la composición por año difiere entre ambos grupos.

### 3.1 ¿Las diferencias se mantienen dentro de un mismo período?

In [ ]:
con_anio = con_anio.assign(periodo=np.where(con_anio.ANIO_PATENTE <= 2020, "2020 o antes", "2021 o después"))
cruce = (con_anio.groupby(["tesoreria", "periodo"]).moroso.agg(n="size", tasa="mean").unstack())
cruce.columns = [f"{a}_{b}" for a, b in cruce.columns]
cruce = cruce.rename(columns={"n_2020 o antes": "n_antes", "n_2021 o después": "n_despues", "tasa_2020 o antes": "tasa_antes", "tasa_2021 o después": "tasa_despues"})
print((cruce.assign(tasa_antes=lambda d: (d.tasa_antes*100).round(1), tasa_despues=lambda d: (d.tasa_despues*100).round(1))
       .sort_values("n_antes", ascending=False)).to_string())
print()
for per, s in con_anio.groupby("periodo"):
    print(per, (tasa_con_intervalo(s, "persona")[["persona", "n", "tasa"]].assign(tasa=lambda d: (d.tasa*100).round(1))).to_string(index=False))

**Lectura.** Las diferencias entre tesorerías cambian mucho según el período. Antofagasta pasa de 91,3 % (patentes de 2020 o antes, n = 92) a 14,7 % (2021 o después, n = 627); su tasa global baja porque la mayoría de sus patentes son recientes. Tarapacá tiene solo 11 patentes de 2020 o antes. Por eso la comparación entre regiones debe hacerse dentro de un mismo período. En cambio, la diferencia por tipo de titular se mantiene en ambos períodos: los titulares naturales presentan mayor morosidad (92,8 % frente a 89,4 % hasta 2020; 74,5 % frente a 60,0 % desde 2021).

## 4. Monto de la patente según situación de pago

In [ ]:
resumen = df.groupby("moroso").UTM.describe(percentiles=[.25, .5, .75])[["count", "25%", "50%", "75%"]]
resumen.index = ["Sin saldo impago", "Con saldo impago"]
print(resumen.round(2).to_string())

**Lectura.** La distribución del monto es muy asimétrica (por eso se usa mediana y rango intercuartílico, y escala logarítmica en el gráfico, que es un violín —densidad— con una caja interior que marca mediana, rango intercuartílico y percentiles 5 a 95). Los cobros con saldo impago tienen una mediana mayor (57,32 UTM, rango intercuartílico 20,00 a 166,40; frente a 28,16 UTM, 8,80 a 90,20). Es una asociación: el saldo impago y el monto no son independientes, porque `SALDO = MONTO − PAGADO` (el monto está en UTM y en pesos).

## 5. Visualizaciones

In [ ]:
import matplotlib; matplotlib.use("Agg")
import runpy; runpy.run_path(str(RAIZ / "F4" / "generar_figuras.py"), run_name="__main__")
from IPython.display import Image, display
for f in ("fig1_tasa_por_anio", "fig2_tasa_por_tesoreria", "fig3_monto_por_situacion"):
    display(Image(filename=str(RAIZ / "F4" / "figuras" / f"{f}.png"), width=720))

## 6. Línea base y modelo exploratorio

Antes de modelar se fija la **línea base**: predecir siempre «moroso» acierta el 77,43 % de las filas de prueba, así que la exactitud sola no basta. Se ajusta una regresión logística con las variables predictoras (se excluyen `SALDO`, `PAGADO`, `LOG_SALDO` y `LOG_PAGADO` por fuga de información). Aquí sí se estandarizan `LOG_MONTO` y `LOG_UTM`, con parámetros aprendidos solo en entrenamiento, porque el modelo lo requiere; el análisis descriptivo se mantiene en UTM y sin escalar.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, precision_score, recall_score, confusion_matrix
tr = pd.read_csv(RAIZ/"data"/"processed"/"entrenamiento_f3.csv"); te = pd.read_csv(RAIZ/"data"/"processed"/"prueba_f3.csv")
def matriz(d, columnas_anio=None):
    X = d[["LOG_MONTO", "LOG_UTM", "anio_faltante", "persona_juridica"] + [c for c in d.columns if c.startswith("teso_")]].copy()
    if columnas_anio is not None:
        anio = d.ANIO_PATENTE.where(d.ANIO_PATENTE > 0, "faltante").astype(str)
        X = pd.concat([X, pd.get_dummies(anio, dtype=int).reindex(columns=columnas_anio, fill_value=0)], axis=1)
    return X
cols_anio = list(pd.get_dummies(tr.ANIO_PATENTE.where(tr.ANIO_PATENTE > 0, "faltante").astype(str)).columns)
filas = []
for nombre, ca in [("Sin año de patente", None), ("Con año de patente", cols_anio)]:
    A, B = matriz(tr, ca), matriz(te, ca)
    sc = StandardScaler().fit(A[["LOG_MONTO", "LOG_UTM"]])           # aprende solo con entrenamiento
    A[["LOG_MONTO", "LOG_UTM"]] = sc.transform(A[["LOG_MONTO", "LOG_UTM"]])
    B[["LOG_MONTO", "LOG_UTM"]] = sc.transform(B[["LOG_MONTO", "LOG_UTM"]])
    m = LogisticRegression(max_iter=2000).fit(A, tr.moroso)
    p = m.predict_proba(B)[:, 1]; y = (p >= 0.5).astype(int)
    filas.append({"modelo": nombre, "AUC": roc_auc_score(te.moroso, p), "exactitud": accuracy_score(te.moroso, y),
                  "exactitud balanceada": balanced_accuracy_score(te.moroso, y),
                  "precisión (moroso)": precision_score(te.moroso, y), "recall (moroso)": recall_score(te.moroso, y)})
    print(nombre, "matriz de confusión [[VN, FP], [FN, VP]]:", confusion_matrix(te.moroso, y).tolist())
linea = {"modelo": "Línea base (siempre moroso)", "AUC": 0.5, "exactitud": te.moroso.mean(), "exactitud balanceada": 0.5,
         "precisión (moroso)": te.moroso.mean(), "recall (moroso)": 1.0}
tabla_modelos = pd.DataFrame([linea] + filas).set_index("modelo")
print(tabla_modelos.round(3).to_string())

In [ ]:
# Figura complementaria (no forma parte de las tres figuras del informe): ROC y matriz de confusión del modelo con año
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
A, B = matriz(tr, cols_anio), matriz(te, cols_anio)
sc = StandardScaler().fit(A[["LOG_MONTO", "LOG_UTM"]])
A[["LOG_MONTO", "LOG_UTM"]] = sc.transform(A[["LOG_MONTO", "LOG_UTM"]]); B[["LOG_MONTO", "LOG_UTM"]] = sc.transform(B[["LOG_MONTO", "LOG_UTM"]])
p = LogisticRegression(max_iter=2000).fit(A, tr.moroso).predict_proba(B)[:, 1]
fpr, tpr, _ = roc_curve(te.moroso, p); cm = confusion_matrix(te.moroso, (p >= .5).astype(int))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.8))
a1.plot(fpr, tpr, color="#1E395E", label=f"Modelo con año (AUC = {roc_auc_score(te.moroso, p):.3f})".replace(".", ","))
a1.plot([0, 1], [0, 1], "--", color="#6B7280", label="Línea base (AUC = 0,500)")
a1.set_xlabel("Tasa de falsos positivos"); a1.set_ylabel("Tasa de verdaderos positivos"); a1.legend(frameon=False, loc="lower right"); a1.set_title("Curva ROC en prueba", loc="left", fontsize=10)
a2.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2): a2.text(j, i, f"{cm[i, j]:,}".replace(",", "."), ha="center", va="center", fontsize=11)
a2.set_xticks([0, 1], ["No moroso", "Moroso"]); a2.set_yticks([0, 1], ["No moroso", "Moroso"]); a2.set_xlabel("Predicho"); a2.set_ylabel("Real"); a2.set_title("Matriz de confusión (umbral 0,5)", loc="left", fontsize=10)
fig.text(.01, .01, "Fuente: Tesorería General de la República, datos abiertos, corte a diciembre de 2025; elaboración propia.", fontsize=7, color="#6B7280")
fig.tight_layout(rect=(0, .04, 1, 1)); fig.savefig(RAIZ / "F4" / "figuras" / "figc_modelo_roc_matriz.png", dpi=150); plt.close()
from IPython.display import Image, display; display(Image(filename=str(RAIZ / "F4" / "figuras" / "figc_modelo_roc_matriz.png"), width=720))

**Lectura.** Con el año de la patente el modelo mejora sobre la línea base (AUC cercano a 0,83 y exactitud balanceada cercana a 0,67), pero **detecta pocos no morosos** (la mayoría se clasifica como moroso). El año es la variable que más aporta. Se evaluó una sola vez en prueba. Queda como limitación que la partición es aleatoria por fila y un mismo titular puede aparecer en ambos conjuntos.

## 7. Mediciones de eficiencia regeneradas

In [ ]:
import medicion
medicion.cargar_base = medicion.cargar_base          # misma base de la Fase 3
tabla = medicion.main()

In [ ]:
# Regenera F3/mediciones.md con los números de ESTA ejecución (informe y repositorio coinciden)
t = tabla.set_index(["comparación", "versión"])
def g(c, v, col): return t.loc[(c, v), col]
f = lambda x, d=6: f"{x:.{d}f}".replace(".", ",")
v1 = g("moroso", "moroso_bucle", "tiempo_s") / g("moroso", "moroso_vectorizado", "tiempo_s")
v2 = g("tasa por tesorería", "tasa_diccionario", "tiempo_s") / g("tasa por tesorería", "tasa_groupby", "tiempo_s")
md = RAIZ / "F3" / "mediciones.md"; txt = md.read_text(encoding="utf-8")
import re
def fila(txt, etiqueta, tiempo, mem):
    return re.sub(rf"(\| {re.escape(etiqueta)}[^|]*\| )[\d,]+( \| )[\d,]+( \|)", lambda m: f"{m.group(1)}{f(tiempo)}{m.group(2)}{f(mem,3)}{m.group(3)}", txt, count=1)
txt = fila(txt, "Bucle fila a fila", g("moroso","moroso_bucle","tiempo_s"), g("moroso","moroso_bucle","memoria_MiB"))
txt = fila(txt, "Vectorizada", g("moroso","moroso_vectorizado","tiempo_s"), g("moroso","moroso_vectorizado","memoria_MiB"))
txt = fila(txt, "Diccionario recorrido a mano", g("tasa por tesorería","tasa_diccionario","tiempo_s"), g("tasa por tesorería","tasa_diccionario","memoria_MiB"))
txt = fila(txt, "`groupby`", g("tasa por tesorería","tasa_groupby","tiempo_s"), g("tasa por tesorería","tasa_groupby","memoria_MiB"))
txt = re.sub(r"es [\d,]+ veces más rápida y usa menos memoria", f"es {f(v1,1)} veces más rápida y usa menos memoria", txt)
txt = re.sub(r"es [\d,]+ veces más rápido, pero", f"es {f(v2,1)} veces más rápido, pero", txt)
md.write_text(txt, encoding="utf-8")
print(f"F3/mediciones.md actualizado: vectorizada {v1:.1f} veces más rápida; groupby {v2:.1f} veces más rápido")

## 8. Validación: pruebas automatizadas

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=RAIZ, capture_output=True, text=True)
print(r.stdout[-400:]); assert r.returncode == 0

## 9. Conclusiones
1. La morosidad no es homogénea: cae de cerca de 90 % a cerca de 65 % entre las patentes de 2020 y las de 2021, y varía de 5 % a 91 % entre tesorerías.
2. El año de la patente es la variable más informativa; el modelo exploratorio mejora la línea base pero no detecta bien a los no morosos.
3. Las mediciones, las pruebas y las cifras de este notebook salen de una misma ejecución, que alimenta el informe y el repositorio.

**Limitaciones.** El archivo es un corte a diciembre de 2025; el cambio de 2021 no se puede explicar con estos datos; la partición no separa por titular; los resultados son asociaciones.